# Data Cleaning and Preparation

## Project Overview
This notebook performs the data cleaning and preparation steps for the Diabetes Hospital Readmission dataset. The goal of this project is to prepare a clean and machine-learning-ready dataset that can later be used to build predictive models for early hospital readmission.

The dataset contains records of diabetic patients admitted to 130 US hospitals between 1999 and 2008. The target variable indicates whether a patient was readmitted to the hospital within 30 days after discharge.

## Objectives of this Notebook
The purpose of this notebook is to transform the raw dataset into a clean dataset suitable for machine learning. The following steps are performed:

1. Load and inspect the raw dataset
2. Handle missing values
3. Remove identifiers and non-informative features
4. Remove constant and near-constant variables
5. Simplify diagnosis codes into broader medical categories
6. Convert age groups into numeric values
7. Handle categorical inconsistencies (race, gender)
8. Simplify medication variables
9. Create new useful features
10. Prepare the target variable for modeling
11. Save the cleaned dataset for further analysis

## Output
The final cleaned dataset is saved to:

`data/processed/diabetes_clean.csv`

This dataset will be used in the next notebook for exploratory data analysis (EDA) and machine learning modeling.

# Data Cleaning and Preparation

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:

df_raw = pd.read_csv("../data/raw/diabetic_data.csv")
df = df_raw.copy()

print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
# Check data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,1.652016e+08,1.026403e+08,12522.0,84961194.0,152388987.0,2.302709e+08,443867222.0
patient_nbr,101766.0,5.433040e+07,3.869636e+07,135.0,23413221.0,45505143.0,8.754595e+07,189502619.0
admission_type_id,101766.0,2.024006e+00,1.445403e+00,1.0,1.0,1.0,3.000000e+00,8.0
discharge_disposition_id,101766.0,3.715642e+00,5.280166e+00,1.0,1.0,1.0,4.000000e+00,28.0
admission_source_id,101766.0,5.754437e+00,4.064081e+00,1.0,1.0,7.0,7.000000e+00,25.0
time_in_hospital,101766.0,4.395987e+00,2.985108e+00,1.0,2.0,4.0,6.000000e+00,14.0
num_lab_procedures,101766.0,4.309564e+01,1.967436e+01,1.0,31.0,44.0,5.700000e+01,132.0
num_procedures,101766.0,1.339730e+00,1.705807e+00,0.0,0.0,1.0,2.000000e+00,6.0
num_medications,101766.0,1.602184e+01,8.127566e+00,1.0,10.0,15.0,2.000000e+01,81.0
number_outpatient,101766.0,3.693572e-01,1.267265e+00,0.0,0.0,0.0,0.000000e+00,42.0


In [6]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [7]:
df["patient_nbr"].nunique()

71518

In [31]:
# Replace '?' with np.nan for consistent missing value handling
df.replace('?', np.nan, inplace=True)

# 2) Basic audit
print("Shape:", df.shape)
print("\nMissing values (top 15):")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.head(15))

print("\nColumns with >80% missing:")
print(missing_pct[missing_pct > 80])

Shape: (101766, 50)

Missing values (top 15):
weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
encounter_id          0.000000
troglitazone          0.000000
tolbutamide           0.000000
pioglitazone          0.000000
rosiglitazone         0.000000
acarbose              0.000000
dtype: float64

Columns with >80% missing:
weight           96.858479
max_glu_serum    94.746772
A1Cresult        83.277322
dtype: float64


In [32]:
cols_to_drop = ["weight", "max_glu_serum", "A1Cresult"]

df = df.drop(columns=cols_to_drop)

print("New shape:", df.shape)

# confirm they are gone
print("Still present?", [c for c in cols_to_drop if c in df.columns])

New shape: (101766, 47)
Still present? []


In [8]:
unique_counts = pd.DataFrame({
    "column": df.columns,
    "unique_values": [df[col].nunique() for col in df.columns]
}).sort_values(by="unique_values", ascending=False)

print(unique_counts)

                      column  unique_values
0               encounter_id         101766
1                patient_nbr          71518
20                    diag_3            789
19                    diag_2            748
18                    diag_1            716
12        num_lab_procedures            118
14           num_medications             75
11         medical_specialty             72
15         number_outpatient             39
16          number_emergency             33
7   discharge_disposition_id             26
17          number_inpatient             21
10                payer_code             17
8        admission_source_id             17
21          number_diagnoses             16
9           time_in_hospital             14
4                        age             10
5                     weight              9
6          admission_type_id              8
13            num_procedures              7
2                       race              5
30                 glipizide    

In [33]:
print("Shape after dropping high-missing columns:", df.shape)

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("\nTop 10 missing columns now:")
print(missing_pct.head(10))

Shape after dropping high-missing columns: (101766, 47)

Top 10 missing columns now:
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
encounter_id          0.000000
tolazamide            0.000000
tolbutamide           0.000000
pioglitazone          0.000000
dtype: float64


In [34]:
cols_to_drop = ["medical_specialty", "payer_code"]

df = df.drop(columns=cols_to_drop)

print("Shape after Step 3:", df.shape)

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("\nTop 10 missing columns now:")
print(missing_pct.head(10))

Shape after Step 3: (101766, 45)

Top 10 missing columns now:
race             2.233555
diag_3           1.398306
diag_2           0.351787
diag_1           0.020636
encounter_id     0.000000
tolazamide       0.000000
tolbutamide      0.000000
pioglitazone     0.000000
rosiglitazone    0.000000
acarbose         0.000000
dtype: float64


In [36]:
df = df.drop(columns=["encounter_id"], errors="ignore")

print("Shape after dropping encounter_id:", df.shape)

Shape after dropping encounter_id: (101766, 44)


In [37]:
df = df.drop(columns=["diag_2", "diag_3"], errors="ignore")

print("Shape after dropping diag_2 and diag_3:", df.shape)

Shape after dropping diag_2 and diag_3: (101766, 42)


In [38]:
# Check columns with only 1 unique value
constant_cols = []

for col in df.columns:
    if df[col].nunique(dropna=False) == 1:
        constant_cols.append(col)

print("Columns with exactly 1 unique value:")
print(constant_cols)

Columns with exactly 1 unique value:
['examide', 'citoglipton']


In [39]:
df = df.drop(columns=["examide", "citoglipton"])

print("Shape after dropping constant columns:", df.shape)

Shape after dropping constant columns: (101766, 40)


In [40]:
near_constant_cols = []

for col in df.columns:
    # Skip numeric columns for now
    if df[col].dtype == "object":
        top_freq = df[col].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq >= 0.995:
            near_constant_cols.append((col, round(top_freq, 4)))

print("Near-constant categorical columns (>=99.5% same value):")
for col, freq in near_constant_cols:
    print(f"{col} → {freq}")

Near-constant categorical columns (>=99.5% same value):
chlorpropamide → 0.9992
acetohexamide → 1.0
tolbutamide → 0.9998
acarbose → 0.997
miglitol → 0.9996
troglitazone → 1.0
tolazamide → 0.9996
glipizide-metformin → 0.9999
glimepiride-pioglitazone → 1.0
metformin-rosiglitazone → 1.0
metformin-pioglitazone → 1.0


In [41]:
cols_to_drop = [col for col, _ in near_constant_cols]

df = df.drop(columns=cols_to_drop)

print("Shape after dropping near-constant columns:", df.shape)

Shape after dropping near-constant columns: (101766, 29)


In [42]:
print("Current columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nNumeric columns:")
print(df.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical columns:")
print(df.select_dtypes(include="object").columns.tolist())

Current columns:
['patient_nbr', 'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'glyburide-metformin', 'change', 'diabetesMed', 'readmitted']

Data types:
object    17
int64     12
Name: count, dtype: int64

Numeric columns:
['patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical columns:
['race', 'gender', 'age', 'diag_1', 'metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'ins

In [43]:
print("Unique diag_1 values:", df["diag_1"].nunique())
print(df["diag_1"].value_counts().head(10))

Unique diag_1 values: 716
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64


In [44]:
# Check how many E/V codes exist
ev_codes = df["diag_1"].astype(str).str.startswith(("E", "V")).sum()
print("Number of E/V codes:", ev_codes)

Number of E/V codes: 1645


In [45]:
def icd_to_float(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    if x.startswith(("E", "V")):
        return np.nan
    try:
        return float(x)
    except:
        return np.nan

def diag_group(x):
    x = icd_to_float(x)
    if pd.isna(x):
        return "Other"
    if 250 <= x < 251:
        return "Diabetes"
    if 390 <= x <= 459 or x == 785:
        return "Circulatory"
    if 460 <= x <= 519 or x == 786:
        return "Respiratory"
    if 520 <= x <= 579 or x == 787:
        return "Digestive"
    if 580 <= x <= 629 or x == 788:
        return "Genitourinary"
    if 800 <= x <= 999:
        return "Injury"
    if 710 <= x <= 739:
        return "Musculoskeletal"
    if 140 <= x <= 239:
        return "Neoplasms"
    return "Other"

df["diag_1_grp"] = df["diag_1"].apply(diag_group)

print(df["diag_1_grp"].value_counts())

diag_1_grp
Circulatory        30437
Other              18193
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Name: count, dtype: int64


In [46]:
df = df.drop(columns=["diag_1"])

print("Shape after removing raw diag_1:", df.shape)

Shape after removing raw diag_1: (101766, 29)


In [9]:
for col in df.columns:
    print("\n", col)
    print(df[col].value_counts(normalize=True).head())


 encounter_id
encounter_id
2278392      0.00001
190792044    0.00001
190790070    0.00001
190789722    0.00001
190786806    0.00001
Name: proportion, dtype: float64

 patient_nbr
patient_nbr
88785891    0.000393
43140906    0.000275
1660293     0.000226
88227540    0.000226
23199021    0.000226
Name: proportion, dtype: float64

 race
race
Caucasian          0.764868
AfricanAmerican    0.193079
Hispanic           0.020474
Other              0.015137
Asian              0.006443
Name: proportion, dtype: float64

 gender
gender
Female             0.537586
Male               0.462384
Unknown/Invalid    0.000029
Name: proportion, dtype: float64

 age
age
[70-80)    0.256156
[60-70)    0.220928
[50-60)    0.169565
[80-90)    0.168986
[40-50)    0.095169
Name: proportion, dtype: float64

 weight
weight
[75-100)     0.417892
[50-75)      0.280576
[100-125)    0.195496
[125-150)    0.045355
[25-50)      0.030341
Name: proportion, dtype: float64

 admission_type_id
admission_type_id
1    0.53053

In [47]:
df["gender"].value_counts()

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

In [48]:
df = df[df["gender"] != "Unknown/Invalid"]

print("Rows after removing invalid gender:", df.shape)

Rows after removing invalid gender: (101763, 29)


In [49]:
df["age"].value_counts().sort_index()

age
[0-10)        161
[10-20)       691
[20-30)      1657
[30-40)      3775
[40-50)      9685
[50-60)     17256
[60-70)     22482
[70-80)     26066
[80-90)     17197
[90-100)     2793
Name: count, dtype: int64

In [50]:
age_map = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
}

df["age"] = df["age"].map(age_map)

print(df["age"].head())
print(df["age"].describe())

0     5
1    15
2    25
3    35
4    45
Name: age, dtype: int64
count    101763.000000
mean         65.966854
std          15.941022
min           5.000000
25%          55.000000
50%          65.000000
75%          75.000000
max          95.000000
Name: age, dtype: float64


In [51]:
print("Missing race values:", df["race"].isna().sum())
print("Percentage:", df["race"].isna().mean() * 100)

Missing race values: 2271
Percentage: 2.2316559063706847


In [52]:
df["race"] = df["race"].fillna("Unknown")

print(df["race"].value_counts())

race
Caucasian          76099
AfricanAmerican    19210
Unknown             2271
Hispanic            2037
Other               1505
Asian                641
Name: count, dtype: int64


In [53]:
df["metformin"].value_counts()

metformin
No        81776
Steady    18345
Up         1067
Down        575
Name: count, dtype: int64

In [54]:
df["insulin"].value_counts()

insulin
No        47380
Steady    30849
Down      12218
Up        11316
Name: count, dtype: int64

In [55]:
med_cols = [
    "metformin","repaglinide","nateglinide","glimepiride",
    "glipizide","glyburide","pioglitazone","rosiglitazone",
    "insulin","glyburide-metformin"
]

print("Medication columns:")
print(med_cols)

Medication columns:
['metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'glyburide-metformin']


In [56]:
for col in med_cols:
    df[col] = df[col].replace({
        "No": 0,
        "Steady": 1,
        "Up": 1,
        "Down": 1
    })

df[med_cols].head()

C:\Users\samir\AppData\Local\Temp\ipykernel_40972\1848045169.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({


,metformin,repaglinide,nateglinide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,insulin,glyburide-metformin
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,1,0,0,0,1,0


In [57]:
df["total_diabetes_meds"] = df[med_cols].sum(axis=1)

df["total_diabetes_meds"].describe()

count    101763.000000
mean          1.174641
std           0.916838
min           0.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           6.000000
Name: total_diabetes_meds, dtype: float64

In [58]:
df["readmitted"].value_counts(normalize=True)

readmitted
NO     0.539106
>30    0.349292
<30    0.111602
Name: proportion, dtype: float64

In [59]:
df["readmitted"] = (df["readmitted"] == "<30").astype(int)

df["readmitted"].value_counts(normalize=True)

readmitted
0    0.888398
1    0.111602
Name: proportion, dtype: float64

In [60]:
print("Final dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes.value_counts())

print("\nTarget distribution:")
print(df["readmitted"].value_counts())

Final dataset shape: (101763, 30)

Data types:
int64     24
object     5
int32      1
Name: count, dtype: int64

Target distribution:
readmitted
0    90406
1    11357
Name: count, dtype: int64


In [62]:
df.to_csv("../data/processed/diabetes_clean.csv", index=False)
print("Saved dataset shape:", df.shape)

Saved dataset shape: (101763, 30)


# Data Cleaning Summary

## Overview
This notebook prepared the Diabetes Hospital Readmission dataset for machine learning. The raw dataset contained missing values, high-cardinality variables, constant features, and categorical inconsistencies that could negatively affect model performance. Several preprocessing steps were applied to improve data quality and model readiness.

## Cleaning Steps Performed

### 1. Handling Missing Values
Columns with extremely high missingness (>80%) were removed:
- `weight`
- `max_glu_serum`
- `A1Cresult`

Columns with moderate missingness were also removed due to limited predictive value:
- `medical_specialty`
- `payer_code`

Small amounts of missing data in `race` were handled by introducing an "Unknown" category.

### 2. Removing Non-informative Columns
Identifier variables and constant features were removed because they do not contribute to prediction:
- `encounter_id`
- `examide`
- `citoglipton`

Additionally, several near-constant medication features were removed to reduce noise and dimensionality.

### 3. Diagnosis Simplification
The diagnosis variable contained hundreds of ICD codes. To reduce dimensionality and improve interpretability, the primary diagnosis (`diag_1`) was grouped into broader medical categories such as:
- Circulatory
- Respiratory
- Digestive
- Diabetes
- Injury
- Neoplasms
- Other

Secondary diagnosis variables (`diag_2`, `diag_3`) were removed to avoid excessive feature sparsity.

### 4. Feature Transformations
Several variables were transformed to make them more suitable for machine learning:
- Age intervals were converted to numeric midpoints.
- Medication states (No, Steady, Up, Down) were simplified into binary indicators (0/1).
- A new feature `total_diabetes_meds` was created to represent the number of diabetes medications used by each patient.

### 5. Target Variable Preparation
The target variable `readmitted` originally contained three categories:
- `NO`
- `>30`
- `<30`

For the purpose of predicting early hospital readmission, the variable was converted into a binary classification target:
- `1` = readmitted within 30 days
- `0` = otherwise

### 6. Final Dataset
After cleaning and feature engineering, the final dataset contains:

- **101,763 observations**
- **30 variables**

This cleaned dataset is suitable for further exploratory analysis and machine learning modeling.

## Next Steps
The cleaned dataset will be used in the next notebook to perform:
- Exploratory Data Analysis (EDA)
- Feature encoding
- Model training and evaluation